# VSLink — Ghép câu: Mức 0 và Mức 1 (chỉ viết prompt)

So sánh trên **cùng tập kiểm tra, cùng nhiễu nhận dạng**:

| Hệ | Mô tả |
|---|---|
| `noi-tho` | Mức 0: nối top-1 theo thứ tự ký |
| `quy-tac` | Mức 0+: đảo trật tự NNKH → tiếng Việt bằng quy tắc, không LLM |
| Gemini / Qwen-2.5-7B / Llama-3.1-8B | Mức 1: 4 điều kiện prompt `zero`, `quy-tac`, `few`, `quy-tac+few` (+ biến thể `--truy-xuat`) |

Thiết kế 4 điều kiện theo SignAlignLM (İnan và cs., ACL Findings 2025).
Cần: GPU T4 (16 GB) cho mô hình 7–8B nén 4-bit. Gemini cần secret `GEMINI_API_KEY`
(Add-ons → Secrets). Llama-3.1 cần secret `HF_TOKEN` và đã chấp nhận giấy phép trên Hugging Face.

In [ ]:
# Lay code (nhanh tinh-nang/ghep-cau). Kaggle: Settings -> Internet ON, Accelerator GPU T4 x2.
!rm -rf vslink && git clone -q --depth 1 -b tinh-nang/ghep-cau https://github.com/HuThnhNg/vslink.git
%cd vslink/nghien-cuu/ghep-cau
!pip install -q sacrebleu

In [ ]:
# (Nen co) Nhieu nhan dang THAT: tai du_doan_test.jsonl (xuat bang xuat_top5_kaggle.py tu notebook
# huan luyen) len Kaggle thanh mot Dataset, "Add Input", roi sua duong dan duoi day.
import glob, shutil
tim = glob.glob('/kaggle/input/**/du_doan_test.jsonl', recursive=True)
if tim:
    shutil.copy(tim[0], 'du-lieu/du_doan_test.jsonl'); print('Dung nhieu THAT:', tim[0])
else:
    print('Chua co du_doan_test.jsonl -> dung nhieu GIA LAP (tam thoi, khong dung cho bao cao cuoi)')
!python tao_du_lieu.py

In [ ]:
import chay_llm
for he in ['noi-tho', 'quy-tac']:
    for dv in ['sach', 'nhieu']:
        chay_llm.main(['--he', he, '--dau-vao', dv])

## Gemini (qua API, không cần GPU)

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['GEMINI_API_KEY'] = UserSecretsClient().get_secret('GEMINI_API_KEY')
except Exception as e:
    print('Khong co GEMINI_API_KEY -> bo qua Gemini', e)
MO_HINH_GEMINI = 'gemini-3.1-flash-lite'   # doi theo ai.google.dev/gemini-api/docs/models
if os.environ.get('GEMINI_API_KEY'):
    he = chay_llm.Gemini(MO_HINH_GEMINI)
    for dk in ['zero', 'quy-tac', 'few', 'quy-tac+few']:
        chay_llm.main(['--he', f'gemini:{MO_HINH_GEMINI}', '--dieu-kien', dk], he=he)
    chay_llm.main(['--he', f'gemini:{MO_HINH_GEMINI}', '--dieu-kien', 'quy-tac+few', '--truy-xuat'], he=he)

## Qwen-2.5-7B-Instruct và Llama-3.1-8B-Instruct (4-bit, chạy trên GPU Kaggle)

In [ ]:
!pip install -q -U bitsandbytes accelerate transformers peft

In [ ]:
import gc, torch, os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass



for ten in ['Qwen/Qwen2.5-7B-Instruct', 'meta-llama/Llama-3.1-8B-Instruct']:
    try:
        he = chay_llm.HF(ten, bon_bit=True)
    except Exception as e:
        print('Khong nap duoc', ten, '->', e); continue
    for dk in ['zero', 'quy-tac', 'few', 'quy-tac+few']:
        chay_llm.main(['--he', f'hf:{ten}', '--dieu-kien', dk], he=he)
    chay_llm.main(['--he', f'hf:{ten}', '--dieu-kien', 'quy-tac+few', '--truy-xuat'], he=he)
    del he; gc.collect(); torch.cuda.empty_cache()

In [ ]:
from IPython.display import Markdown
Markdown(open('du-lieu/ket-qua/tong-ket.md', encoding='utf-8').read())

In [ ]:
# Tai ket qua ve (du-lieu/ket-qua/*.jsonl co ca cau tra loi tho cua tung mo hinh de phan tich loi)
!cd du-lieu && zip -qr /kaggle/working/ket-qua-muc1.zip ket-qua